# Counters — Flip-Flops That Count

A counter is a register whose state advances through a fixed sequence on each clock edge. This notebook draws the **flip-flop chain**, the **clock-aligned bit waveforms**, and the **circular state diagram** for each counter type, contrasting asynchronous ripple delay against clean synchronous timing.

$$\text{state}_{k+1} = g(\text{state}_k)\big|_{CLK\uparrow}$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

ON, OFF = '#c0392b', '#b0b0b0'
def wcol(b): return ON if b else OFF
def wlw(b):  return 2.4 if b else 1.2

def clock(n, period=4):
    t = np.arange(n)
    return ((t // (period//2)) % 2).astype(int)
def rising_edges(clk):
    return [i for i in range(1,len(clk)) if clk[i-1]==0 and clk[i]==1]

def state_diagram(ax, states, current, labels=None, title=''):
    """Draw states on a circle with directed transitions; highlight current."""
    m = len(states)
    ang = np.linspace(np.pi/2, np.pi/2 - 2*np.pi, m, endpoint=False)
    R = 1.0
    xs, ys = R*np.cos(ang), R*np.sin(ang)
    for i in range(m):
        j = (i+1) % m
        a = FancyArrowPatch((xs[i], ys[i]), (xs[j], ys[j]),
                            connectionstyle='arc3,rad=0.25',
                            arrowstyle='->', mutation_scale=12,
                            color='#aaa', lw=1.2, zorder=1)
        ax.add_patch(a)
    for i, s in enumerate(states):
        cur = (s == current)
        ax.add_patch(Circle((xs[i], ys[i]), 0.22,
                            fc='#c0392b' if cur else '#eef2f7',
                            ec='#34495e', lw=1.5, zorder=2))
        lab = labels[i] if labels else format(s, 'b')
        ax.text(xs[i], ys[i], lab, ha='center', va='center', fontsize=8,
                color='white' if cur else '#333', weight='bold', zorder=3)
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=9)

print('primitives ready')


primitives ready


## Asynchronous Ripple Counter — Delay Accumulates

Each T flip-flop toggles on the previous stage's output, so bit $i$ only flips after bit $i-1$ has already flipped. The toggles **ripple** down the chain, adding one gate delay per stage. The waveform shows the staggered edges — the higher bits lag visibly.

$$Q_0:\ \div 2,\quad Q_1:\ \div 4,\quad Q_2:\ \div 8,\ \dots$$


In [2]:
def ripple_counter(nbits, delay_ns, step):
    M=2**nbits; count=step % M
    bits=[(count>>b)&1 for b in range(nbits)]
    n = 64
    CLK = clock(n, 4); edges = rising_edges(CLK)
    t = np.arange(n)
    bits_wave = np.zeros((nbits, n), dtype=int)
    prev = CLK
    for b in range(nbits):
        q = 0; col = np.zeros(n, dtype=int)
        pe = [i for i in range(1,n) if prev[i-1]==1 and prev[i]==0]
        for i in range(n):
            if i in pe: q ^= 1
            col[i] = q
        bits_wave[b] = col; prev = col
    fig = plt.figure(figsize=(9, 1.0*(nbits+1)+3))
    # --- FF chain schematic: each FF clocked by previous output ---
    axs = fig.add_axes([0.04, 0.66, 0.92, 0.3]); axs.axis('off')
    axs.set_xlim(0, nbits*2.2+0.5); axs.set_ylim(0,2.2)
    prev_out_x = None
    for b in range(nbits):
        x = 0.4 + b*2.2; bit=bits[b]
        axs.add_patch(Rectangle((x,0.6),1.3,1.0, fc='#fdecea' if bit else '#eef2f7', ec='#34495e', lw=1.5, zorder=2))
        axs.text(x+0.65,1.35,f'T-FF Q{b}',ha='center',fontsize=8,color='#555')
        axs.text(x+0.65,0.95,str(bit),ha='center',va='center',fontsize=13,weight='bold',color=wcol(bit))
        axs.plot([x,x+0.16,x],[0.7,0.8,0.9],color='#34495e',lw=1.0)
        if b==0:
            axs.annotate('',xy=(x,0.8),xytext=(x-0.4,0.8),arrowprops=dict(arrowstyle='->',color='#34495e',lw=1.4))
            axs.text(x-0.4,0.45,'CLK',fontsize=8,color='#34495e')
        else:
            # clocked by previous stage output (the ripple link)
            axs.annotate('',xy=(x,0.8),xytext=(prev_out_x,0.4),
                         arrowprops=dict(arrowstyle='->',color='#e67e22',lw=1.4,connectionstyle='arc3,rad=-0.3'))
        prev_out_x = x+1.3
    axs.text(nbits*2.2/2,1.95,f'count={count} ({format(count,f"0{nbits}b")})  -- each FF clocked by the previous output (ripple)',ha='center',fontsize=9,weight='bold')
    # --- skew waveform ---
    base_y=0.58
    axw0=fig.add_axes([0.08,base_y,0.86,0.08])
    axw0.step(t, CLK, where='post', color='#34495e', lw=2); axw0.set_yticks([]); axw0.set_ylabel('CLK',rotation=0,ha='right'); axw0.grid(True,alpha=0.3); axw0.set_xticklabels([])
    colors = ['#c0392b','#2471a3','#27ae60','#e67e22','#8e44ad']
    for b in range(nbits):
        axb=fig.add_axes([0.08, base_y-0.1*(b+1), 0.86, 0.08])
        shift = (b+1) * delay_ns * 0.05
        axb.step(t + shift, bits_wave[b], where='post', color=colors[b % 5], lw=2)
        axb.set_yticks([]); axb.set_ylabel(f'Q{b} (÷{2**(b+1)})', rotation=0, ha='right'); axb.grid(True,alpha=0.3)
        if b<nbits-1: axb.set_xticklabels([])
        else: axb.set_xlabel('time (each stage lags by accumulated delay)')
    plt.show()

w_rb = widgets.IntSlider(value=3, min=2, max=4, description='bits:')
w_rd = widgets.FloatSlider(value=4.0, min=0.0, max=10.0, step=0.5, description='delay/stage:')
w_rs = widgets.IntSlider(value=0, min=0, max=15, description='count step:')
display(widgets.VBox([w_rb, w_rd, w_rs]),
        widgets.interactive_output(ripple_counter, {'nbits': w_rb, 'delay_ns': w_rd, 'step': w_rs}))


Output()

## Synchronous Counter — The Flip-Flop Chain, Stepped Through Time

Every flip-flop shares the **same clock**; each toggle input is the AND of all lower bits ($T_i = Q_0 Q_1 \cdots Q_{i-1}$). Move the **clock-pulse slider**: the FF chain redraws with the current count lit on the flip-flops, the toggle-enable wires light when their AND condition holds, and the clock waveform marks the active edge. The state diagram tracks the same step.

$$T_i = Q_0 \cdot Q_1 \cdots Q_{i-1}$$


In [ ]:
def sync_counter(nbits, step):
    M = 2**nbits
    count = step % M
    bits = [(count >> b) & 1 for b in range(nbits)]   # bits[0]=LSB
    fig = plt.figure(figsize=(9.2, 5.4))
    # --- FF chain schematic (top) ---
    ax = fig.add_axes([0.04, 0.55, 0.92, 0.42]); ax.axis('off')
    ax.set_xlim(0, nbits*2.2 + 0.5); ax.set_ylim(0, 3)
    for b in range(nbits):                 # draw LSB on the left
        x = 0.4 + b*2.2
        bit = bits[b]
        fc = '#fdecea' if bit else '#eef2f7'
        ax.add_patch(Rectangle((x, 1.0), 1.3, 1.3, fc=fc, ec='#34495e', lw=1.6, zorder=2))
        ax.text(x+0.65, 2.05, f'T-FF Q{b}', ha='center', fontsize=8, color='#555', zorder=3)
        ax.text(x+0.65, 1.5, str(bit), ha='center', va='center', fontsize=15, weight='bold', color=wcol(bit), zorder=3)
        # clock triangle
        ax.plot([x, x+0.18, x],[1.1, 1.22, 1.34], color='#34495e', lw=1.1, zorder=3)
        # toggle enable = AND of all lower bits (LSB always toggles)
        t_en = 1 if b==0 else (1 if all(bits[j]==1 for j in range(b)) else 0)
        ax.text(x+0.65, 2.55, f'T={t_en}', ha='center', fontsize=8, color=wcol(t_en), weight='bold')
        # carry/AND wire from previous stages into this T
        if b>0:
            ax.annotate('', xy=(x+0.65, 2.4), xytext=(x-0.55, 2.4),
                        arrowprops=dict(arrowstyle='->', color=wcol(t_en), lw=wlw(t_en)))
        # output down
        ax.plot([x+0.65, x+0.65],[1.0, 0.6], color=wcol(bit), lw=wlw(bit))
    # shared clock rail
    ax.plot([0.0, nbits*2.2], [0.3, 0.3], color='#34495e', lw=1.5)
    for b in range(nbits):
        x = 0.4 + b*2.2
        ax.plot([x, x],[0.3, 1.1], color='#34495e', lw=1.0)
    ax.text(0.0, 0.05, 'shared CLK', fontsize=8, color='#34495e')
    ax.text(nbits*2.2/2, 2.85, f'count = {count}  ({format(count, f"0{nbits}b")})  -- step {step}',
            ha='center', fontsize=9.5, weight='bold')
    # --- clock waveform pointer (middle) ---
    axc = fig.add_axes([0.06, 0.40, 0.6, 0.1])
    nt=32; CLK=clock(nt,4); edges=rising_edges(CLK); t=np.arange(nt)
    axc.step(t,CLK,where='post',color='#34495e',lw=1.4)
    axc.set_ylim(-0.3,1.3); axc.set_yticks([]); axc.set_ylabel('CLK',rotation=0,ha='right')
    if step>0 and step<=len(edges): axc.axvline(edges[(step-1)%len(edges)],color='#8e44ad',lw=2.2)
    axc.set_xlabel('time (active edge marked)', fontsize=8)
    # --- state diagram (right) ---
    axs = fig.add_axes([0.66, 0.06, 0.32, 0.44])
    state_diagram(axs, list(range(M)), count, title=f'mod-{M} sequence')
    plt.show()

w_sb = widgets.IntSlider(value=3, min=2, max=4, description='bits:')
w_step = widgets.IntSlider(value=0, min=0, max=15, description='clock pulse:')
display(widgets.VBox([w_sb, w_step]),
        widgets.interactive_output(sync_counter, {'nbits': w_sb, 'step': w_step}))


## Mod-N Counter — Reset Before the Natural Wrap

Detecting count $= N$ and clearing the register makes a counter that cycles $0 \dots N-1$ regardless of the natural power-of-two range. A mod-10 (BCD) counter is the classic decimal digit. The diagram shows the truncated cycle and the reset edge back to zero.


In [ ]:
def mod_n(N, current):
    n = 40
    CLK = clock(n, 4); edges = rising_edges(CLK)
    t = np.arange(n)
    cnt = np.zeros(n, dtype=int); c = 0
    for i in range(n):
        if i in edges:
            c += 1
            if c >= N: c = 0
        cnt[i] = c
    fig = plt.figure(figsize=(9,3.4))
    ax1 = fig.add_subplot(1,2,1)
    ax1.step(t, cnt, where='post', color='#c0392b', lw=2)
    ax1.set_ylabel('count'); ax1.set_xlabel('time'); ax1.grid(True, alpha=0.3)
    ax1.axhline(N-1, color='#888', ls='--', lw=1); ax1.set_yticks(range(N))
    ax1.set_title(f'mod-{N}: wraps {N-1} -> 0', fontsize=9)
    ax2 = fig.add_subplot(1,2,2)
    state_diagram(ax2, list(range(N)), current % N, title=f'mod-{N} cycle')
    plt.tight_layout(); plt.show()

w_N = widgets.IntSlider(value=10, min=3, max=12, description='modulus N:')
w_mc = widgets.IntSlider(value=0, min=0, max=11, description='highlight:')
display(widgets.VBox([w_N, w_mc]),
        widgets.interactive_output(mod_n, {'N': w_N, 'current': w_mc}))


## Ring vs Johnson — Shift Registers With Feedback

A **ring counter** feeds the last output straight back to the first: a single 1 circulates, giving $N$ distinct states. A **Johnson** (twisted-ring) counter feeds back the *inverted* output, giving $2N$ states. Both are shift registers closing the loop differently — the direct tie-in to the previous notebook.


In [ ]:
def ring_johnson(N, mode, step):
    if mode == 'ring':
        reg = [1] + [0]*(N-1)
        seq = [reg[:]]
        for _ in range(2*N):
            reg = [reg[-1]] + reg[:-1]; seq.append(reg[:])
        states = [int(''.join(map(str, s)), 2) for s in seq[:N]]
        period = N
    else:  # johnson
        reg = [0]*N
        seq = [reg[:]]
        for _ in range(2*N + 1):
            reg = [1-reg[-1]] + reg[:-1]; seq.append(reg[:])
        states = [int(''.join(map(str, s)), 2) for s in seq[:2*N]]
        period = 2*N
    cur_reg = seq[step % period]
    fig = plt.figure(figsize=(9,3.4))
    ax1 = fig.add_subplot(1,2,1); ax1.axis('off')
    ax1.set_xlim(0, N+1); ax1.set_ylim(0,2)
    for i, b in enumerate(cur_reg):
        ax1.add_patch(Rectangle((i+0.1, 0.7), 0.8, 0.6,
                      fc='#fdecea' if b else '#eef2f7', ec='#34495e', lw=1.4))
        ax1.text(i+0.5, 1.0, str(b), ha='center', va='center', fontsize=12,
                 weight='bold', color=wcol(b))
    fb = cur_reg[-1] if mode=='ring' else 1-cur_reg[-1]
    ax1.annotate(f'fb={fb}', xy=(0.1,1.6), fontsize=9, color=wcol(fb), weight='bold')
    ax1.annotate('', xy=(0.5,1.35), xytext=(N,1.6),
                 arrowprops=dict(arrowstyle='->', color=wcol(fb),
                 connectionstyle='arc3,rad=-0.4', lw=1.5))
    ax1.set_title(f'{mode} counter, step {step}  (period {period})', fontsize=9)
    ax2 = fig.add_subplot(1,2,2)
    labels = [format(s, f'0{N}b') for s in states]
    cur_val = int(''.join(map(str, cur_reg)), 2)
    state_diagram(ax2, states, cur_val, labels=labels, title=f'{period} states')
    plt.tight_layout(); plt.show()

w_rn = widgets.IntSlider(value=4, min=3, max=5, description='bits N:')
w_mode = widgets.Dropdown(options=['ring','johnson'], value='johnson', description='type:')
w_step = widgets.IntSlider(value=0, min=0, max=12, description='step:')
display(widgets.VBox([w_rn, w_mode, w_step]),
        widgets.interactive_output(ring_johnson, {'N': w_rn, 'mode': w_mode, 'step': w_step}))


## Gray Code Counter — One Bit Changes at a Time

Binary counting can flip several bits at once ($011 \to 100$), risking decode glitches. Gray code orders states so **exactly one bit** changes per step. The plot compares the per-transition bit-flip count of binary vs Gray across a full cycle.

$$G = B \oplus (B \gg 1)$$


In [ ]:
def gray_compare(nbits):
    M = 2**nbits
    binary = list(range(M))
    gray = [b ^ (b >> 1) for b in binary]
    def flips(seq):
        return [bin(seq[i] ^ seq[(i+1) % M]).count('1') for i in range(M)]
    fb, fg = flips(binary), flips(gray)
    fig, axes = plt.subplots(1,2, figsize=(9,3.2))
    x = range(M)
    axes[0].bar(x, fb, color='#2471a3', alpha=0.8)
    axes[0].set_title(f'binary: up to {nbits} bits flip at once'); axes[0].set_xlabel('transition')
    axes[0].set_ylabel('bits changed'); axes[0].set_ylim(0, nbits+0.5); axes[0].grid(True, alpha=0.3)
    axes[1].bar(x, fg, color='#c0392b', alpha=0.8)
    axes[1].set_title('Gray: always exactly 1'); axes[1].set_xlabel('transition')
    axes[1].set_ylim(0, nbits+0.5); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print('binary:', [format(b, f'0{nbits}b') for b in binary])
    print('gray:  ', [format(g, f'0{nbits}b') for g in gray])
    print('max bits flipped — binary:', max(fb), ' gray:', max(fg))

w_gb = widgets.IntSlider(value=3, min=2, max=5, description='bits:')
display(w_gb, widgets.interactive_output(gray_compare, {'nbits': w_gb}))
